In [3]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter

# =============1.准备文档================
with open('../resources/评估.md', 'r', encoding='utf-8') as f:
    doc = f.read()

# =============2.切分文档================
# 切分依据，这里是按照三级标题
headers_to_split_on = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]

# 创建切分器
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on,
    strip_headers=False # 在正文中是否去除标题，False，不去除
)

# 切分文档, 返回的是 list[Document]
docs = markdown_splitter.split_text(doc)

for i, doc in enumerate(docs, start=1):
    doc.id = f'doc_{i}'

# 1.初始化向量模型
ollama_embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 可选：减少维度以节省存储
)

# 2.初始化向量数据库
vector_store = InMemoryVectorStore(embedding=ollama_embeddings)
# 保存文档到向量数据库
ids = vector_store.add_documents(docs)
print(f"=======================ids==========================")
print(ids)
# 用户问题
query = "论语中教育的目的是什么"

# 相似度检索，返回值是list[Document]
results = vector_store.similarity_search(
    query=query,
    k = 5,
)
print(f"=======================docs==========================")
# 打印结果
for doc in results:
    print(doc.model_dump_json(indent=2))
# 相似度搜索并返回分数，结果是list[tuple[Document, float]]
results = vector_store.similarity_search_with_score(
    query,
    k = 5,
)
print(f"=======================score==========================")
# 遍历结果，
for doc, score in results:
    print(f"==========score: {score}=============")
    print(doc.model_dump_json(indent=2))

=======================ids==========================
['doc_1', 'doc_2', 'doc_3', 'doc_4', 'doc_5', 'doc_6', 'doc_7', 'doc_8', 'doc_9', 'doc_10', 'doc_11', 'doc_12', 'doc_13', 'doc_14', 'doc_15', 'doc_16', 'doc_17', 'doc_18', 'doc_19', 'doc_20', 'doc_21']
=======================docs==========================
{
  "id": "doc_3",
  "metadata": {
    "h1": "第一章 教育概述",
    "h2": "第一节 中外教育家及其教育思想",
    "h3": "（三）孔子及《论语》主要思想"
  },
  "page_content": "### （三）孔子及《论语》主要思想  \n**教育作用**：庶、富、教；性相近，习相远。  \n**教育对象**：“有教无类”，教育民主思想。  \n**教育目的**：以完善人格为教育的首要目的，培养士和君子。  \n**教学内容**：文、行、忠、义。  \n**教学过程**：学、思、习、行。  \n**教学原则**：因材施教、启发诱导、学思结合、谦虚笃实。  \n**教师观**：其身正，不令而行；其身不正，虽令不从。",
  "type": "Document"
}
{
  "id": "doc_4",
  "metadata": {
    "h1": "第一章 教育概述",
    "h2": "第一节 中外教育家及其教育思想",
    "h3": "（四）孟子主要思想"
  },
  "page_content": "### （四）孟子主要思想  \n**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  \n**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  \n**教学原则**：循序渐进，专心有恒。",
  "type": "Document"
}
{
  "id": "doc_5",
  "metadata": {
    "h1": "第一章 教